# Day 05 — Exception Handling + Retry + logging
**Module 01: Python + Async + FastAPI for LLM Engineering**  
Vidya Sankalp | Applied GenAI Engineering

> **What you'll learn:** `try/except` · catching specific exceptions · `finally` · `raise` · built-in exception types · the `logging` module · log levels · retry with exponential backoff

> **Connection to Day 02:** The `while` retry loop from Day 02 is now production-grade — wrapped in `try/except` with logging at every step.

> **Note on OOP:** Custom exception *classes* (using `class MyError(Exception)`) require OOP knowledge covered in Day 08. Today we use Python's **built-in exception types** only — `ValueError`, `KeyError`, `json.JSONDecodeError`, etc.


---


## 1 — Imports


In [ ]:
import json       # json.JSONDecodeError — raised when JSON is invalid
import logging    # structured logging — replaces print() in production
import time       # time.sleep() — pause between retries


---


## 2 — What is an Exception?

An **exception** is an error that occurs at runtime — when something unexpected happens that Python cannot handle by itself.

Without handling exceptions, one bad CSV row or one invalid LLM response crashes your entire service. With `try/except`, you handle the error gracefully and keep running.

### Common built-in exceptions you will see every day

| Exception | When it's raised | LLM engineering example |
|-----------|-----------------|------------------------|
| `KeyError` | Dict key doesn't exist | `row['customer_id']` when column is missing |
| `ValueError` | Value has wrong format | `int('abc')` — can't convert |
| `TypeError` | Wrong type passed | `len(None)` — None has no length |
| `FileNotFoundError` | File path doesn't exist | `open('prompts/missing.txt')` |
| `json.JSONDecodeError` | String is not valid JSON | `json.loads('not json')` |
| `ZeroDivisionError` | Division by zero | `total / count` when count is 0 |
| `IndexError` | List index out of range | `items[10]` on a 3-item list |
| `AttributeError` | Object has no such attribute | `None.strip()` |


---


## 3 — try / except — Basic Syntax

```python
try:
    # the risky code — might raise an exception
    result = int(user_input)
except ValueError:
    # runs ONLY if the try block raised a ValueError
    print('Not a number')
```

**Python checks the `except` clauses top to bottom — most specific first.**  
If no exception is raised, the `except` block is skipped entirely.


In [5]:
int('3.14')

ValueError: invalid literal for int() with base 10: '3.14'

In [2]:
# ── 1. Catching a single exception type ─────────────────────
def safe_to_int(value):
    """Convert value to int. Returns None if conversion fails."""
    try:
        return int(value)   # may raise ValueError or TypeError
    except ValueError:
        # ValueError: value has wrong format → int('abc') → ValueError
        print(f"  ValueError: '{value}' cannot be converted to int")
        return None

print(safe_to_int('1001'))  # 1001  — no exception
print(safe_to_int('abc'))   # None  — ValueError caught
print(safe_to_int('3.14'))  # None  — ValueError caught (float str → int fails)


1001
  ValueError: 'abc' cannot be converted to int
None
  ValueError: '3.14' cannot be converted to int
None


In [6]:
# ── 2. Catching multiple exception types ─────────────────────
def get_customer_name(customer: dict, key: str):
    """Get a field from customer dict safely."""
    try:
        return customer[key].strip().title()   # KeyError if key missing, AttributeError if None
    except KeyError:
        # KeyError: key doesn't exist in the dict
        print(f"  KeyError: key '{key}' not found in customer dict")
        return 'Unknown'
    except AttributeError:
        # AttributeError: value exists but is None — None.strip() fails
        print(f"  AttributeError: value for '{key}' is None")
        return 'Unknown'

customer = {'customer_id': '1001', 'name': 'prudhvi akella', 'email': None}

print(get_customer_name(customer, 'name'))      # 'Prudhvi Akella'
print(get_customer_name(customer, 'email'))     # AttributeError — None.strip()
print(get_customer_name(customer, 'city'))      # KeyError — key missing


Prudhvi Akella
  AttributeError: value for 'email' is None
Unknown
  KeyError: key 'city' not found in customer dict
Unknown


In [ ]:
def get_customer_name(customer: dict, key: str):
    """Get a field from customer dict safely."""
    try:
        return customer[key].strip().title()   # KeyError if key missing, AttributeError if None
    except Exception:
        print(f" There is an issue with key:'{key}' either its not exist or value is None")

In [8]:
# ── 3. Catching multiple types in one except line ────────────
def parse_csv_field(value: str, target_type: type):
    """Convert a CSV string field to target_type."""
    try:
        return target_type(value)
    except (ValueError, TypeError):
        # (ValueError, TypeError) in a tuple — catches both in one clause
        print(f"  Could not convert '{value}' to {target_type.__name__}")
        return None

print(parse_csv_field('205.21', float))   # 205.21
print(parse_csv_field('abc',    float))   # None  — ValueError
print(parse_csv_field(None,     float))   # None  — TypeError


205.21
  Could not convert 'abc' to float
None
  Could not convert 'None' to float
None


In [7]:
float(None)

TypeError: float() argument must be a string or a real number, not 'NoneType'

---


## 4 — Reading the Exception Message with `as`

Add `as e` to capture the exception object. This gives you the error message and lets you log it.

```python
except ValueError as e:
    print(f'Error: {e}')   # e contains the error message string
```


In [9]:
import json
# except ExceptionType as e — e holds the exception object
# str(e) or f'{e}' gives the error message

bad_json = '{"category": "TRACK_ORDER", missing_quote: "high"}'  # invalid JSON

try:
    parsed = json.loads(bad_json)
except json.JSONDecodeError as e:
    # json.JSONDecodeError: raised when json.loads() gets invalid JSON
    # e.msg        → short error message
    # e.lineno     → line number where error occurred
    # e.colno      → column number where error occurred
    print(f"JSONDecodeError: {e}")
    print(f"  message : {e.msg}")
    print(f"  position: line {e.lineno}, column {e.colno}")
    parsed = {}   # safe fallback — empty dict
except Exception as e:
    print(f"Exception: {e}")

print(f"parsed = {parsed}")


JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 29 (char 28)
  message : Expecting property name enclosed in double quotes
  position: line 1, column 29
parsed = {}


---


## 5 — `finally` — Always Runs

`finally` runs **no matter what** — whether the `try` block succeeded, raised an exception, or was caught by `except`.  
Use it for **cleanup**: closing files, recording timing, releasing locks.

```python
try:
    result = risky_operation()
except SomeError as e:
    handle_error(e)
finally:
    cleanup()   # ALWAYS runs — success or failure
```


In [1]:
import time
# Python function from the time module that returns a high-resolution performance counter used for measuring short durations with very high precision.
# It represents the number of seconds elapsed since an arbitrary reference point chosen by the operating system.
# The absolute value itself has no meaning.Only the difference between two readings is useful.
time.perf_counter()

554492.5123845

In [10]:
import time

def parse_with_timing(raw_response: str) -> dict:
    """Parse LLM JSON and always log how long it took."""
    start = time.perf_counter()   # start timer before the risky code
    result = {}

    try:
        cleaned = raw_response.strip()
        if cleaned.startswith('```'):
            cleaned = cleaned.split('\n', 1)[-1].rsplit('```', 1)[0].strip()
        result = json.loads(cleaned)
        print(f"  Parse succeeded: {result}")

    except json.JSONDecodeError as e:
        print(f"  Parse FAILED: {e.msg}")
        result = {}   # safe empty fallback

    finally:
        # finally ALWAYS runs — we always want to record the timing
        end = time.perf_counter()
        elapsed_ms = round((end - start) * 1000, 2)
        print(f"  Elapsed: {elapsed_ms}ms  (finally block — always runs)")

    return result

# Test 1: valid JSON
print("Valid JSON:")
parse_with_timing('{"category": "TRACK_ORDER", "confidence": "high"}')
print()

# Test 2: invalid JSON
print("Invalid JSON:")
parse_with_timing('{bad json here}')


Valid JSON:
  Parse succeeded: {'category': 'TRACK_ORDER', 'confidence': 'high'}
  Elapsed: 0.02ms  (finally block — always runs)

Invalid JSON:
  Parse FAILED: Expecting property name enclosed in double quotes
  Elapsed: 0.02ms  (finally block — always runs)


{}

---


In [16]:
raw_response = '{bad json here}'
try:
    cleaned = raw_response.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.split('\n', 1)[-1].rsplit('```', 1)[0].strip()
    result = json.loads(cleaned)
    print(f"  Parse succeeded: {result}")
except json.JSONDecodeError as e:
    print(f"  Parse FAILED: {e.msg}")
    result = {}
print("XYZ")
print("ABC")
print("Hello World")

  Parse FAILED: Expecting property name enclosed in double quotes
XYZ
ABC
Hello World


## 6 — `raise` — Re-raising and Raising Exceptions

| Pattern | When to use |
|---------|------------|
| `raise` | Re-raise the current exception (keeps original traceback) |
| `raise ValueError('message')` | Raise a new exception with a custom message |
| `raise ValueError('msg') from e` | Raise a new exception, chain it to the original |

Use `raise` when the current function cannot handle the error itself and the **caller** needs to know about it.


In [23]:
def validate_customer_id(customer_id: str) -> int:
    """Convert customer_id string to int, raise ValueError with a clear message if invalid."""
    try:
        cid = int(customer_id)
    except ValueError as e:
        # Don't silently return None — raise a clear error so the caller knows
        raise ValueError(f"customer_id must be a number, got: '{customer_id}'") from e
        # 'from e' chains the original exception — Python shows both in the traceback

    if cid <= 0:
        # Raise a ValueError ourselves when the value is logically invalid
        raise ValueError(f"customer_id must be positive, got: {cid}")

    return cid

# Test: valid id
print(validate_customer_id('1001'))   # 1001

# Test: invalid id
try:
    validate_customer_id('abc')
except ValueError as e:
    print(f"Caught: {e}")
except Exception as e:
    print(f"Caught: {e}")

# Test: negative id
try:
    validate_customer_id('-5')
except ValueError as e:
    print(f"Caught: {e}")
except Exception as e:
    print(f"Caught: {e}")


1001
Caught: customer_id must be a number, got: 'abc'
Caught: customer_id must be positive, got: -5


### Exception Propagation — How Exceptions Travel Up the Call Stack

When a function raises an exception and **does not catch it**, the exception travels up to whoever called that function.
It keeps travelling up until something catches it — or the program crashes.

```
build_customer_prompt('abc')        ← Level 3 — catches the exception here
  └─ load_customer('abc')            ← Level 2 — no try/except, exception passes through
       └─ parse_customer_id('abc')   ← Level 1 — raises ValueError here
            └─ raises ValueError
       ↑ propagates up automatically
  ↑ propagates up automatically
  caught!
```

**Each function only needs to handle what it can meaningfully fix.**
If a function cannot fix the error, it should let it propagate — not swallow it silently.


In [25]:
# ── Level 1 — deepest function — raises the exception ───────
def parse_customer_id(raw_id):
    """Convert raw string to a valid positive integer customer ID."""
    try:
        cid = int(raw_id)
    except ValueError:
        # raise travels UP to whoever called parse_customer_id
        raise ValueError(f"customer_id must be a number, got: '{raw_id}'")
    if cid <= 0:
        raise ValueError(f"customer_id must be positive, got: {cid}")
    return cid


# ── Level 2 — middle function — no try/except ────────────────
def load_customer(raw_id):
    """
    Load a customer by ID.
    No try/except here — this function cannot decide what to do
    about a bad ID. Let the exception propagate to the caller.
    """
    customer_id = parse_customer_id(raw_id)   # may raise ValueError
    # If parse_customer_id raises, we NEVER reach the line below
    # Python immediately jumps up the call stack looking for a handler
    return {'customer_id': customer_id, 'name': 'Prudhvi Akella', 'city': 'Rajahmundry'}


# ── Level 3 — top-level function — catches and handles ────────
def build_customer_prompt(raw_id):
    """
    Build a prompt string for the LLM using customer data.
    This is the right place to catch — we know what a sensible
    fallback looks like at this level.
    """
    try:
        customer = load_customer(raw_id)   # calls load_customer → calls parse_customer_id
        return f"Customer: {customer['name']} (id={customer['customer_id']})"

    except ValueError as e:
        # This exception was raised inside parse_customer_id (3 levels down)
        # but it propagated all the way up here because no one caught it sooner
        return f"Could not build prompt — bad customer ID: {e}"


# ── TRACE: valid input ────────────────────────────────────────
# build_customer_prompt('1001')
#   → load_customer('1001')
#       → parse_customer_id('1001') → returns 1001  (no exception)
#   → returns {'customer_id': 1001, 'name': ...}
# → returns 'Customer: Prudhvi Akella (id=1001)'
print(build_customer_prompt('1001'))

# ── TRACE: invalid input ─────────────────────────────────────
# build_customer_prompt('abc')
#   → load_customer('abc')
#       → parse_customer_id('abc')
#           → int('abc') raises ValueError
#           → re-raised as ValueError("customer_id must be a number, got: 'abc'")
#       ← exception propagates — load_customer has no handler
#   ← exception propagates — build_customer_prompt catches it
# → returns 'Could not build prompt — bad customer ID: ...'
print(build_customer_prompt('abc'))
print(build_customer_prompt('-5'))

Customer: Prudhvi Akella (id=1001)
Could not build prompt — bad customer ID: customer_id must be a number, got: 'abc'
Could not build prompt — bad customer ID: customer_id must be positive, got: -5


---


## 7 — The `logging` Module

### Why not just use `print()`?

| | `print()` | `logging` |
|-|-----------|----------|
| Shows timestamp | No | Yes |
| Shows severity level | No | Yes (INFO, WARNING, ERROR) |
| Can be turned off | No — always prints | Yes — set level to filter |
| Goes to file + console | No | Yes — multiple handlers |
| Production-ready | No | Yes |

### Log levels — in order of severity

| Level | Number | When to use |
|-------|--------|------------|
| `DEBUG` | 10 | Detailed diagnostics — only during development |
| `INFO` | 20 | Normal operation — function started/completed |
| `WARNING` | 30 | Unexpected but handled — missing field, slow response |
| `ERROR` | 40 | Something failed — exception caught |
| `CRITICAL` | 50 | Serious failure — service about to stop |

> Setting level to `WARNING` means only WARNING, ERROR, and CRITICAL messages appear — DEBUG and INFO are filtered out.


In [8]:
import logging

# basicConfig sets up the root logger — call ONCE at the top of your application
# format = what each log line looks like
# level  = minimum severity to display (INFO means DEBUG is hidden)
logging.basicConfig(
    level=logging.DEBUG,
    format='%(name)s - %(asctime)s -  %(levelname)s - %(message)s',
    force=True,   # force=True lets us reconfigure in Jupyter (normally call once)
)

log = logging.getLogger(__name__)   # get a logger named after this module

# The five log levels
log.debug(   'detailed info, only shown in development')
log.info(    'normal operation: function started, request received')
log.warning( 'unexpected but handled: missing CSV column')
log.error(   'something failed: json.JSONDecodeError caught')
log.critical('serious failure: database unreachable')


__main__ - 2026-06-05 08:09:37,430 -  DEBUG - detailed info, only shown in development
__main__ - 2026-06-05 08:09:37,431 -  INFO - normal operation: function started, request received
__main__ - 2026-06-05 08:09:37,431 -  WARNING - unexpected but handled: missing CSV column
__main__ - 2026-06-05 08:09:37,432 -  ERROR - something failed: json.JSONDecodeError caught
__main__ - 2026-06-05 08:09:37,432 -  CRITICAL - serious failure: database unreachable


In [9]:
import time
import json
# ── Logging in practice — with exception handling ────────────
def safe_parse_llm_json(raw_response: str) -> dict:
    """
    Parse LLM JSON response with logging at each step.
    No OOP — uses only built-in json.JSONDecodeError.
    """
    log.info(f"Parsing LLM response ({len(raw_response)} chars)")
    start = time.perf_counter()

    try:
        # Step 1: strip markdown fences if present
        cleaned = raw_response.strip()
        if cleaned.startswith('```'):
            cleaned = cleaned.split('\n', 1)[-1].rsplit('```', 1)[0].strip()
            log.debug('Stripped markdown fences from response')

        # Step 2: parse JSON string → Python dict
        # json.loads() raises json.JSONDecodeError if string is not valid JSON
        parsed = json.loads(cleaned)
        log.debug(f"Parsed successfully. Keys: {list(parsed.keys())}")
        return parsed

    except json.JSONDecodeError as e:
        # json.JSONDecodeError is a built-in exception — no OOP needed
        log.error(f"JSON parse failed at line {e.lineno} col {e.colno}: {e.msg}")
        log.error(f"Raw response was: {raw_response[:100]}")
        return {}   # return empty dict — caller checks with .get()

    finally:
        elapsed_ms = round((time.perf_counter() - start) * 1000, 2)
        log.info(f"Parse finished in {elapsed_ms}ms")

# Demo
print("\n--- Test 1: valid JSON ---")
result = safe_parse_llm_json('{"category": "TRACK_ORDER", "confidence": "high"}')
print(f"Result: {result}")

print("\n--- Test 2: fenced JSON ---")
result2 = safe_parse_llm_json('```json\n{"intent": "CANCEL"}\n```')
print(f"Result: {result2}")

print("\n--- Test 3: invalid JSON ---")
result3 = safe_parse_llm_json('{bad json here}')
print(f"Result: {result3}")


__main__ - 2026-06-05 08:09:48,337 -  INFO - Parsing LLM response (49 chars)
__main__ - 2026-06-05 08:09:48,338 -  DEBUG - Parsed successfully. Keys: ['category', 'confidence']
__main__ - 2026-06-05 08:09:48,338 -  INFO - Parse finished in 0.31ms
__main__ - 2026-06-05 08:09:48,339 -  INFO - Parsing LLM response (32 chars)
__main__ - 2026-06-05 08:09:48,339 -  DEBUG - Stripped markdown fences from response
__main__ - 2026-06-05 08:09:48,339 -  DEBUG - Parsed successfully. Keys: ['intent']
__main__ - 2026-06-05 08:09:48,339 -  INFO - Parse finished in 0.49ms
__main__ - 2026-06-05 08:09:48,340 -  INFO - Parsing LLM response (15 chars)
__main__ - 2026-06-05 08:09:48,340 -  ERROR - JSON parse failed at line 1 col 2: Expecting property name enclosed in double quotes
__main__ - 2026-06-05 08:09:48,340 -  ERROR - Raw response was: {bad json here}
__main__ - 2026-06-05 08:09:48,341 -  INFO - Parse finished in 0.5ms



--- Test 1: valid JSON ---
Result: {'category': 'TRACK_ORDER', 'confidence': 'high'}

--- Test 2: fenced JSON ---
Result: {'intent': 'CANCEL'}

--- Test 3: invalid JSON ---
Result: {}


---


## 8 — Applying try/except to Real Data: CSV Row Parsing

CSV data from `DictReader` always gives strings — and columns can be missing or malformed.  
Wrap conversion in `try/except` so one bad row doesn't crash processing of thousands of good rows.


In [ ]:
def parse_customer_csv_row(row: dict) -> dict:
    """
    Parse and validate a single row from customers.csv.
    Returns a typed dict on success, empty dict on failure.
    Logs a warning for any row that fails — never crashes.
    """
    try:
        return {
            'customer_id': int(row['customer_id']),          # str → int
            'name'       : f"{row['first_name']} {row['last_name']}".strip(),
            'email'      : row['email'].lower().strip(),
            'city'       : row['city'],
        }

    except KeyError as e:
        # KeyError: a required column is missing from this CSV row
        # Usually means the CSV schema changed — log it and skip the row
        log.warning(f"Missing column in customer row: {e}. Row: {row}")
        return {}

    except ValueError as e:
        # ValueError: a column value can't be converted to the expected type
        # e.g. int('N/A') → ValueError
        log.warning(f"Type conversion failed for customer row: {e}. Row: {row}")
        return {}


# Demo
print("--- Valid row ---")
valid_row = {'customer_id': '1001', 'first_name': 'Prudhvi', 'last_name': 'Akella',
             'email': '  PRUDHVI@Example.COM  ', 'city': 'Rajahmundry'}
print(parse_customer_csv_row(valid_row))

print("\n--- Missing column (KeyError) ---")
missing_col = {'customer_id': '1002', 'first_name': 'Ravi'}   # last_name, email, city missing
print(parse_customer_csv_row(missing_col))

print("\n--- Bad type (ValueError) ---")
bad_type = {'customer_id': 'N/A', 'first_name': 'Ananya', 'last_name': 'Sharma',
            'email': 'a@b.com', 'city': 'Bangalore'}
print(parse_customer_csv_row(bad_type))


---


## 9 — Retry with Exponential Backoff + Logging

This is the Day 02 `while` retry loop now made production-grade:  
- `try/except` inside the `while` loop catches specific errors  
- `logging` records every attempt, wait, and outcome  
- Some errors (auth failure) should **not** be retried — `raise` immediately  
- Some errors (rate limit, timeout) should be retried after a wait  

```
attempt=0 → API call fails (rate limit) → wait 0.1s → attempt=1
attempt=1 → API call fails (timeout)    → wait 0.2s → attempt=2
attempt=2 → API call succeeds           → return response
```


In [10]:
def _simulate_llm_call(prompt: str, attempt: int) -> str:
    """
    Simulates a real LLM API call for demo purposes.
    attempt 0 → ConnectionError (rate limit)
    attempt 1 → TimeoutError
    attempt 2 → success
    """
    if attempt == 0:
        raise ConnectionError('HTTP 429: Too Many Requests')
    elif attempt == 1:
        raise TimeoutError('Request timed out after 30s')
    else:
        return '{"category": "TRACK_ORDER", "confidence": "high"}'

In [17]:
import time
import traceback

def call_llm_api_with_retry(
    prompt: str,
    model: str = 'gpt-4o',
    max_retries: int = 3,
) -> str:
    """
    Call an LLM API with retry logic and logging.
    Different errors get different treatment:
      - ConnectionError   → retry after wait (recoverable)
      - TimeoutError      → retry after wait (recoverable)
      - PermissionError   → raise immediately (auth failure — retrying won't help)
      - json.JSONDecodeError → retry with cleaner prompt instruction
    """
    attempt    = 0
    last_error = None

    # ── ITERATION TRACE (3 retries, succeeds on attempt 3):
    # attempt=0 → _simulate_llm_call raises ConnectionError
    #             wait = 0.1 * (2**0) = 0.1s
    #             attempt = 1
    #
    # attempt=1 → _simulate_llm_call raises TimeoutError
    #             wait = 0.1 * (2**1) = 0.2s
    #             attempt = 2
    #
    # attempt=2 → _simulate_llm_call succeeds
    #             returns '{"category": "TRACK_ORDER", ...}'
    while attempt < max_retries:
        try:
            log.info(f"LLM call attempt {attempt + 1}/{max_retries} | model={model}")
            response = _simulate_llm_call(prompt, attempt)
            log.info(f"Succeeded on attempt {attempt + 1}")
            return response   # success — exit loop immediately

        except PermissionError as e:
            # Auth errors are NOT retryable — fail immediately
            # 'raise' re-raises the current exception with its original traceback
            log.error(f"Auth failed — check your API key: {e}")
            raise

        except (ConnectionError, TimeoutError) as e:
            # Transient errors — wait and retry with exponential backoff
            wait = 0.1 * (2 ** attempt)   # 0.1s, 0.2s, 0.4s, 0.8s
            log.warning(f"Attempt {attempt + 1} failed ({type(e).__name__}). Waiting {wait:.1f}s")
            log.warning(traceback.format_exc())
            time.sleep(wait)
            time.sleep(wait)
            last_error = e

        except json.JSONDecodeError as e:
            # Bad JSON — add stricter instruction and retry
            log.warning(f"LLM returned invalid JSON. Retrying with stricter instruction.")
            prompt     = prompt + '\n\nIMPORTANT: Respond ONLY with valid JSON.'
            last_error = e

        attempt += 1   # MUST increment — otherwise infinite loop

    # All retries exhausted
    log.error(f"All {max_retries} attempts failed. Last error: {last_error}")
    raise RuntimeError(f"LLM call failed after {max_retries} attempts") from last_error


---


In [18]:
# Demo
print("--- Retry demo ---")
try:
    response = call_llm_api_with_retry('Where is order #3042?', max_retries=3)
    print(f"Final response: {response}")
except RuntimeError as e:
    print(f"All retries exhausted: {e}")

INFO     LLM call attempt 1/3 | model=gpt-4o
WARNING  Attempt 1 failed (ConnectionError). Waiting 0.1s
WARNING  Traceback (most recent call last):
  File "/var/folders/dq/76c6b9rx7tv5fhypn7n6fx800000gn/T/ipykernel_4097/969837533.py", line 34, in call_llm_api_with_retry
    response = _simulate_llm_call(prompt, attempt)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/dq/76c6b9rx7tv5fhypn7n6fx800000gn/T/ipykernel_4097/3686699563.py", line 9, in _simulate_llm_call
    raise ConnectionError('HTTP 429: Too Many Requests')
ConnectionError: HTTP 429: Too Many Requests



--- Retry demo ---


INFO     LLM call attempt 2/3 | model=gpt-4o
WARNING  Attempt 2 failed (TimeoutError). Waiting 0.2s
WARNING  Traceback (most recent call last):
  File "/var/folders/dq/76c6b9rx7tv5fhypn7n6fx800000gn/T/ipykernel_4097/969837533.py", line 34, in call_llm_api_with_retry
    response = _simulate_llm_call(prompt, attempt)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/dq/76c6b9rx7tv5fhypn7n6fx800000gn/T/ipykernel_4097/3686699563.py", line 11, in _simulate_llm_call
    raise TimeoutError('Request timed out after 30s')
TimeoutError: Request timed out after 30s

INFO     LLM call attempt 3/3 | model=gpt-4o
INFO     Succeeded on attempt 3


Final response: {"category": "TRACK_ORDER", "confidence": "high"}


## 10 — Filtering Log Levels

In production you set the level to `WARNING` so only problems appear — not every INFO or DEBUG message.  
In development you use `DEBUG` to see everything.


In [16]:
# Set level to WARNING — only WARNING, ERROR, CRITICAL appear
logging.basicConfig(
    level=logging.ERROR,
    format='%(levelname)-8s %(message)s',
    force=True,
)
log = logging.getLogger(__name__)

log.debug(  'DEBUG   — NOT shown (below WARNING level)')
log.info(   'INFO    — NOT shown (below WARNING level)')
log.warning('WARNING — shown')
log.error(  'ERROR   — shown')

print()
# Reset to DEBUG for the rest of the notebook
logging.basicConfig(level=logging.DEBUG, format='%(levelname)-8s %(message)s', force=True)
log = logging.getLogger(__name__)


ERROR    ERROR   — shown


---


## Summary — Day 05

| Concept | Key rule / syntax |
|---------|-------------------|
| `try/except` | Wrap risky code. `except` runs only if that error type is raised. |
| Catch specific first | `except ValueError` before `except Exception` — most specific first |
| `except (A, B)` | Catch multiple types in one clause |
| `except Error as e` | `e` holds the exception object and message |
| `finally` | ALWAYS runs — use for cleanup, timing, closing resources |
| `raise` | Re-raise current exception (keeps traceback) |
| `raise ValueError('msg')` | Raise a new exception with a clear message |
| `raise X from e` | Chain new exception to original |
| `KeyError` | Dict key missing — `row['col']` when column absent |
| `ValueError` | Wrong format — `int('abc')` |
| `json.JSONDecodeError` | Invalid JSON — `json.loads('bad')` |
| `FileNotFoundError` | File path doesn't exist |
| `logging.basicConfig()` | Call ONCE at startup. Sets format and minimum level. |
| Log levels | DEBUG < INFO < WARNING < ERROR < CRITICAL |
| `log.info()` / `log.warning()` / `log.error()` | Use the right level for the right event |
| Retry pattern | `while attempt < max_retries` + `try/except` + `time.sleep(0.1 * 2**attempt)` |
| Don't retry auth errors | `raise` immediately on `PermissionError` — retrying won't help |
| OOP custom exceptions | Covered in Day 08 after OOP is introduced |

**Next:** Day 06 — Decorators and Higher-Order Functions  
`@decorator` syntax · `functools.wraps` · timing decorators · retry as a decorator
